# Question:
### What is the chance of getting a flush that scores more than 300 chips in the first round without discarding in one hand in balatro? 

# simulation:
### joint prob **:3**
#### not including flush house, straight flush, royal flush, or flush house <!-- move to question later -->
#### red deck, white stake <!-- move to question later -->

# Description:
### random process: drawing 8 random cards w/o replacement -- Deck: standard 52 cards
### one trial: 8 cards are drawn. 
### possible outcomes: A: hand is a regular flush, B: hand scores > 300
### compound event: whether there are at least 5 cards of the same suit(no order) and whether those cards get a round score of 300 or more

### A && B
### not mutually exlusive, they can happen(and are being tested when they happen) at the same time 
### independency: yes?????? diff chips & mult

# Analysis:

In [62]:
import pandas as pd
import numpy.random as rand
from numpy import array
from fractions import Fraction
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import math

In [25]:
# Starting hand size: 8
# Flush desc: 5 cards of the same suit

In [53]:
# Random Number Generator
rng = rand.default_rng()

# Constants
HAND_SIZE = 8
BASE_CHIPS = 35
BASE_MULT = 4
TARGET_SCORE = 300

# Define suits and ranks
suits = ['Spades', 'Hearts', 'Diamonds', 'Clubs']
ranks = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'Jack', 'Queen', 'King', 'Ace']

# Rank order for straight checks
rank_order = {r: i for i, r in enumerate(ranks)}

# Base chip values for Balatro card ranks
card_chips = {
    '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, '10': 10,
    'Jack': 10, 'Queen': 10, 'King': 10, 'Ace': 11
}

# 1. Create a standard 52-card deck
deck = [{'rank': r, 'suit': s} for s in suits for r in ranks]

In [54]:
def draw_cards(size, data):
    # draw cards without replacement
    drawn_cards = rng.choice(data, size=size, replace=False).tolist()
    return drawn_cards

In [55]:
def is_straight_flush(combo):
    # Check if 5 cards form a straight (consecutive ranks or Ace-low A-2-3-4-5)
    indices = sorted([rank_order[c['rank']] for c in combo])
    
    # Standard straight
    if len(set(indices)) == 5 and indices[-1] - indices[0] == 4:
        return True
    
    # Ace-low straight (Ace, 2, 3, 4, 5) -> indices [0, 1, 2, 3, 12]
    if sorted(indices) == [0, 1, 2, 3, 12]:
        return True
        
    return False

In [58]:
def evaluate_hand(hand):
    suit_groups = {}
    for card in hand:
        suit_groups.setdefault(card['suit'], []).append(card)
    
    max_score = 0
    has_valid_flush = False
    
    for suit, cards in suit_groups.items():
        if len(cards) >= 5:
            for combo in itertools.combinations(cards, 5):
                # Exclude Straight Flushes (including Royal Flushes) and Flush Houses
                if is_straight_flush(combo):
                    continue
                
                has_valid_flush = True
                chips = BASE_CHIPS + sum(card_chips[c['rank']] for c in combo)
                score = chips * BASE_MULT
                if score > max_score:
                    max_score = score
                    
    scores_over_300 = max_score >= TARGET_SCORE
    return has_valid_flush, scores_over_300, max_score

In [59]:
# Run 100 trials
trials_data = []
for trial in range(1, 101):
    hand = draw_cards(8, deck)
    
    has_flush, scores_over_300, max_score = evaluate_hand(hand)
    
    hand_str = ", ".join([f"{c['rank']}{c['suit'][0]}" for c in hand])
    
    trials_data.append({
        "Trial": trial,
        "Drawn Hand": hand_str,
        "If hand contains flush": has_flush,
        "If hand scores over 300": scores_over_300,
        "Max Flush Score": max_score if has_flush else 0
    })

# Convert results into a Pandas DataFrame
df = pd.DataFrame(trials_data)

In [60]:
# Display summary and first few rows
print(f"Total Trials: {len(df)}")
print(f"Flushes Found: {df['If hand contains flush'].sum()}")
print(f"Standard Flushes Scoring > 300: {df['If hand scores over 300'].sum()}")
print("\nFirst 10 Rows:")
print(df.head(10).to_string(index=False))

Total Trials: 100
Flushes Found: 8
Standard Flushes Scoring > 300: 2

First 10 Rows:
 Trial                                 Drawn Hand  If hand contains flush  If hand scores over 300  Max Flush Score
     1      5S, QueenD, 9H, 8S, KingC, 3D, 4C, 6D                   False                    False                0
     2      4H, 9S, AceC, 7C, 8H, 10S, 3C, QueenS                   False                    False                0
     3          4H, 6S, 9H, 4C, 5S, 5C, 5H, KingS                   False                    False                0
     4 9S, QueenC, 6S, QueenD, 10C, 2C, KingC, 3S                   False                    False                0
     5 10D, AceS, 2C, 9H, JackC, JackD, 7C, KingD                   False                    False                0
     6 4D, 9D, 3D, JackD, 7S, QueenC, KingH, AceD                    True                    False              288
     7        3D, 3H, 5D, 4S, 5S, 8C, QueenS, 10S                   False                    False     

In [61]:
flush_trials = df[df['If hand contains flush'] == True]

for _, row in flush_trials.iterrows():
    print(f"Trial {row['Trial']}:")
    print(f"  Hand: {row['Drawn Hand']}")
    print(f"  Scores over 300: {row['If hand scores over 300']}")
    print(f"  Max Flush Score: {row['Max Flush Score']}")

Trial 6:
  Hand: 4D, 9D, 3D, JackD, 7S, QueenC, KingH, AceD
  Scores over 300: False
  Max Flush Score: 288
Trial 27:
  Hand: JackH, 3C, KingH, KingD, 4H, AceH, 5C, 3H
  Scores over 300: False
  Max Flush Score: 292
Trial 29:
  Hand: 9H, JackH, JackD, 4H, 3S, QueenH, 6H, 9S
  Scores over 300: False
  Max Flush Score: 296
Trial 30:
  Hand: KingH, 5H, 8H, AceH, AceC, 5S, JackS, 10H
  Scores over 300: True
  Max Flush Score: 316
Trial 41:
  Hand: 4S, 9D, 3D, 2D, 7H, 7C, KingD, 7D
  Scores over 300: False
  Max Flush Score: 264
Trial 42:
  Hand: KingD, 9D, 8H, 8C, QueenD, 4D, AceC, 2D
  Scores over 300: False
  Max Flush Score: 280
Trial 67:
  Hand: 9S, 4D, 4H, KingS, QueenS, 5S, 7C, 10S
  Scores over 300: True
  Max Flush Score: 316
Trial 94:
  Hand: 7D, 3D, KingD, QueenD, 4S, 5H, 9D, KingS
  Scores over 300: False
  Max Flush Score: 296


In [65]:
# Empirical Probability
total_trials = len(df)
empirical_count = df['If hand scores over 300'].sum()
empirical_prob = empirical_count / total_trials

In [66]:
# Theoretical Probability
total_hands = math.comb(52, 8)

def compute_exact_theoretical():
    total_qualifying = 0
    ranks_list = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'Jack', 'Queen', 'King', 'Ace']
    rank_idx = {r: i for i, r in enumerate(ranks_list)}
    
    for c_s in range(0, 9):
        for c_h in range(0, 9 - c_s):
            for c_d in range(0, 9 - c_s - c_h):
                c_c = 8 - c_s - c_h - c_d
                counts = [c_s, c_h, c_d, c_c]
                if not any(c >= 5 for c in counts):
                    continue
                
                ways = 1
                for count in counts:
                    if count < 5:
                        ways *= math.comb(13, count)
                    else:
                        valid_k = 0
                        for suit_cards in itertools.combinations(ranks_list, count):
                            has_300_flush = False
                            for combo5 in itertools.combinations(suit_cards, 5):
                                indices = sorted([rank_idx[r] for r in combo5])
                                is_sf = (len(set(indices)) == 5 and indices[-1] - indices[0] == 4) or (sorted(indices) == [0, 1, 2, 3, 12])
                                if not is_sf and sum(card_chips[r] for r in combo5) >= 40:
                                    has_300_flush = True
                                    break
                            if has_300_flush:
                                valid_k += 1
                        ways *= valid_k
                total_qualifying += ways
    return total_qualifying

exact_qualifying = compute_exact_theoretical()
theoretical_prob = exact_qualifying / total_hands

In [67]:
print("=== EMPIRICAL PROBABILITY ===")
print(f"Trials Scoring >= 300: {empirical_count} / {total_trials}")
print(f"Empirical Probability: {empirical_prob:.4f} ({empirical_prob * 100:.2f}%)\n")

print("=== THEORETICAL PROBABILITY ===")
print(f"Total Possible 8-Card Hands: {total_hands:,}")
print(f"Hands with Regular 300+ Flush: {exact_qualifying:,}")
print(f"Theoretical Probability: {theoretical_prob:.6f} ({theoretical_prob * 100:.4f}%)")

=== EMPIRICAL PROBABILITY ===
Trials Scoring >= 300: 2 / 100
Empirical Probability: 0.0200 (2.00%)

=== THEORETICAL PROBABILITY ===
Total Possible 8-Card Hands: 752,538,150
Hands with Regular 300+ Flush: 17,285,368
Theoretical Probability: 0.022969 (2.2969%)
